In [ ]:
import torch

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from model_v2_compatible import SeqNN

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
model0 = SeqNN()
model0.load_state_dict(torch.load("/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Hsieh2019_mESC/models/Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth", map_location=device))
model0.eval()

In [ ]:
model1 = SeqNN()
model1.load_state_dict(torch.load("/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Hsieh2019_mESC/models/Akita_v2_mouse_Hsieh2019_mESC_model1_finetuned.pth", map_location=device))
model1.eval()

In [ ]:
model2 = SeqNN()
model2.load_state_dict(torch.load("/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Bonev2017_ncx_NPC/models/Akita_v2_mouse_Bonev2017_ncx_NPC_model0_finetuned.pth", map_location=device))
model2.eval()

In [ ]:
model3 = SeqNN()
model3.load_state_dict(torch.load("/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Bonev2017_ncx_NPC/models/Akita_v2_mouse_Bonev2017_ncx_NPC_model1_finetuned.pth", map_location=device))
model3.eval()

In [ ]:
model4 = SeqNN()
model4.load_state_dict(torch.load("/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Hsieh2019_mESC/models/Akita_v2_mouse_Hsieh2019_mESC_model2_finetuned.pth", map_location=device))
model4.eval()

In [ ]:
model5 = SeqNN()
model5.load_state_dict(torch.load("/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Hsieh2019_mESC/models/Akita_v2_mouse_Hsieh2019_mESC_model3_finetuned.pth", map_location=device))
model5.eval()

In [ ]:
model6 = SeqNN()
model6.load_state_dict(torch.load("/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Bonev2017_ncx_NPC/models/Akita_v2_mouse_Bonev2017_ncx_NPC_model2_finetuned.pth", map_location=device))
model6.eval()

In [ ]:
model7 = SeqNN()
model7.load_state_dict(torch.load("/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Bonev2017_ncx_NPC/models/Akita_v2_mouse_Bonev2017_ncx_NPC_model3_finetuned.pth", map_location=device))
model7.eval()

In [ ]:
import pandas as pd

In [ ]:
bin_size = 2048
cropping_applied = 64
padding_bins = 2
padding = padding_bins * bin_size

# 256 +- 25 (50 bins in the center)
slice_0_bins = [256]
slice_0_start = (min(slice_0_bins) + cropping_applied - padding_bins) * bin_size
slice_0_end = (max(slice_0_bins) + 1 + cropping_applied + padding_bins) * bin_size

In [ ]:
edit_start = (min(slice_0_bins) + cropping_applied) * bin_size
edit_end = (max(slice_0_bins) + 1 + cropping_applied) * bin_size

In [ ]:
# chrom = "chr11"
# start = 65677312
# end = 66988032

# chrom = "chr1"
# start = 37799936
# end = 39110656

chrom = "chr3"
start = 38524928
end = 39835648

In [ ]:
X = torch.load(f"/scratch1/smaruj/generate_genomic_boundary/ohe_X/fold0/{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)

In [ ]:
x_bar_slice_0 = torch.load(f"/scratch1/smaruj/generate_cell_type_specific_features/mESC_weak_ncx_NPC_strong_results_sum/{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)

In [ ]:
with torch.no_grad():
    og_pred_model0 = model0(X)
    og_pred_model2 = model2(X)

In [ ]:
X_new = X.clone()

In [ ]:
import numpy as np

In [ ]:
X_new_np = X_new.detach().cpu().numpy()

In [ ]:
np.save(f"/scratch1/smaruj/generate_cell_type_specific_features/mESC_weak_ncx_NPC_strong_results_sum/{chrom}_{start}_{end}_X_og.npy", X_new_np)

In [ ]:
X_new[:, :, edit_start:edit_end] = x_bar_slice_0

In [ ]:
X_np = X_new.detach().cpu().numpy()

In [ ]:
np.save(f"/scratch1/smaruj/generate_cell_type_specific_features/mESC_weak_ncx_NPC_strong_results_sum/{chrom}_{start}_{end}_X_mod.npy", X_np)

In [ ]:
with torch.no_grad():
    pred_model0 = model0(X_new)
    pred_model1 = model1(X_new)
    pred_model2 = model2(X_new)
    pred_model3 = model3(X_new)
    pred_model4 = model4(X_new)
    pred_model5 = model5(X_new)
    pred_model6 = model6(X_new)
    pred_model7 = model7(X_new)

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(og_pred_model0, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(og_pred_model2, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(pred_model0, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(pred_model1, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(pred_model2, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(pred_model3, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(pred_model4, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(pred_model5, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(pred_model6, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(pred_model7, matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("fountain_difference.svg", format='svg')
plt.show()